# wf2020_riskcap_full_bc_replay_v1

Purpose: convert the `wf2020_risk_cap_reference_target_screen_v1` selected `riskcap_full_replacement` target contract into a file-backed BC official replay, then test whether it actually moves the retained-model frontier toward the required walk-forward gate.

Governance:
- Baselines: canonical `v19reg_v22_budget_usage_bull_guard_v1`; direct comparison `wf2020_risk_cap_reference_target_screen_v1`; performance comparison `bull_reentry_three_fold_checkpoint_metric_replay_v1`; reference floor `wf2020_reference_floor_gate_calibration_v1`.
- Role: `diagnose`.
- Target windows: `wf_2020`, `wf_2022`, `wf_2023_2024`.
- Accepted trade-off: MaxDD must stay below 20%; `wf_2020` must beat the calibrated risk-cap reference floor; `wf_2022` and `wf_2023_2024` must keep +30pp relative return.
- Changed layer: `model_selection`, `training_target`, `bc_warm_start`, `official_replay`.
- Hypothesis: the learnable risk-cap replacement target can reduce the wf2020 drawdown/return frontier enough to justify a return-preserving hybrid target or retained training branch.
- Acceptance: all three folds evaluated with no target/test overlap; selected BC official replay passes fold-specific return gates and MaxDD <20%.
- Failure mode: the full replacement is learnable but too conservative; if return underparticipates, do not retain an actor and design a return-preserving hybrid target instead.
- Closest success: `bull_reentry_three_fold_checkpoint_metric_replay_v1` passed `wf_2022` and `wf_2023_2024` under calibrated gates.
- Closest failure: `wf2020_reference_floor_gate_calibration_v1` showed the prior bull/reentry path remained short of the wf2020 risk floor.
- Why not duplicate: this notebook executes the newly selected `riskcap_full_replacement` contract through file-backed official replay, not just BC label fit or target distribution screening.


In [1]:

from __future__ import annotations

import json
import math
import random
import sys
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

ROOT = Path.cwd()
while not (ROOT / 'AGENTS.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.state_conditioned_objective import daily_path_metrics, max_drawdown_interval_metrics
from src.soxl_quant.app.state_objective_bc import (
    StateObjectiveBCSampleWeightConfig,
    StateObjectiveBCTrainingConfig,
    build_state_objective_bc_dataset_from_frame,
    evaluate_bc_actor_slice_fit,
    fit_state_objective_bc_actor_from_frame,
)
from src.soxl_quant.backtest import backtest_profile
from src.soxl_quant.compat import patch_gym_compat
from src.soxl_quant.config import load_profile
from src.soxl_quant.core.metric_reconciliation import build_warm_start_replay_segment, slice_warm_start_daily_path
from src.soxl_quant.data import build_dataset, feature_columns_for_packs, flatten_feature_frame, init_qlib
from src.soxl_quant.settings import LOOKBACK, QLIB_DATA_PATH

patch_gym_compat()

EXPERIMENT = 'wf2020_riskcap_full_bc_replay_v1'
CARD = ROOT / 'runtime/model_selection/card' / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)
SOURCE_CARD = ROOT / 'runtime/model_selection/card/wf2020_risk_cap_reference_target_screen_v1'
TARGETS_PATH = SOURCE_CARD / 'riskcap_full_replacement_combined_target_contract.csv'
GATE_CONTRACT_PATH = ROOT / 'runtime/model_selection/card/wf2020_reference_floor_gate_calibration_v1/training_gate_contract.json'
PREVIOUS_REPLAY_PATH = ROOT / 'runtime/model_selection/card/bull_reentry_three_fold_checkpoint_metric_replay_v1/checkpoint_selection_by_fold.csv'
PROFILE_NAME = 'sparse_release_bc_ppo_screen_v1'
SEED = 20260526
TARGET_MAX_DD = 20.0

WALK_FORWARD_WINDOWS = {
    'wf_2020': {
        'target_fold': 'wf_2020_train',
        'train': ('2010-03-11', '2019-12-31'),
        'valid': ('2020-01-01', '2020-12-31'),
        'test': ('2020-01-01', '2020-12-31'),
    },
    'wf_2022': {
        'target_fold': 'wf_2022_train',
        'train': ('2010-03-11', '2020-12-31'),
        'valid': ('2021-01-01', '2021-12-31'),
        'test': ('2022-01-01', '2022-12-31'),
    },
    'wf_2023_2024': {
        'target_fold': 'wf_2023_2024_train',
        'train': ('2010-03-11', '2022-12-31'),
        'valid': ('2023-01-01', '2023-12-31'),
        'test': ('2024-01-01', '2024-12-31'),
    },
}

BC_TRAIN_CONFIG = StateObjectiveBCTrainingConfig(steps=200, lr=0.001, batch_size=128, seed=SEED)
BC_WEIGHT_CONFIG = StateObjectiveBCSampleWeightConfig(
    rebound_multiplier=4.0,
    downtrend_multiplier=2.0,
    release_gate_multiplier=10.0,
    ordinary_multiplier=1.0,
    objective_weight_power=0.0,
    class_balanced_beta=0.0,
    weight_floor=0.05,
    weight_cap=30.0,
    normalize_mean=True,
)
SLICE_COLS = (
    'risk_cap_reference_active_state',
    'risk_cap_reference_buy_state',
    'risk_cap_reference_sell_state',
    'release_gate_state',
    'is_downtrend_state',
    'drawdown_control_state',
)


def json_default(value: Any) -> Any:
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, (float, np.floating)):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, Path):
        return str(value)
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass
    raise TypeError(f'Object of type {type(value).__name__} is not JSON serializable')


def to_json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): to_json_safe(v) for k, v in value.items()}
    if isinstance(value, list):
        return [to_json_safe(v) for v in value]
    if isinstance(value, tuple):
        return [to_json_safe(v) for v in value]
    if isinstance(value, (float, np.floating)):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, Path):
        return str(value)
    return value


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(
        json.dumps(to_json_safe(payload), indent=2, sort_keys=True, default=json_default, allow_nan=False) + '\n',
        encoding='utf-8',
    )


def set_seed(seed: int) -> None:
    random.seed(int(seed))
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))


def datetime_index_frame(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    if isinstance(out.index, pd.MultiIndex):
        if 'instrument' in out.index.names:
            out = out.droplevel('instrument')
        else:
            out = out.reset_index(level=list(range(1, out.index.nlevels)), drop=True)
    out.index = pd.DatetimeIndex(pd.to_datetime(out.index))
    return out.sort_index()


def datetime_index_series(series: pd.Series) -> pd.Series:
    out = series.copy()
    if isinstance(out.index, pd.MultiIndex):
        if 'instrument' in out.index.names:
            out = out.droplevel('instrument')
        else:
            out.index = out.index.get_level_values(0)
    out.index = pd.DatetimeIndex(pd.to_datetime(out.index))
    return out.sort_index()


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [2]:

profile = load_profile(PROFILE_NAME)
feature_columns = feature_columns_for_packs(profile.feature.packs, profile.name)
source_summary = json.loads((SOURCE_CARD / 'summary.json').read_text(encoding='utf-8'))
gate_contract = json.loads(GATE_CONTRACT_PATH.read_text(encoding='utf-8'))
targets = pd.read_csv(TARGETS_PATH, parse_dates=['datetime'])
previous_replay = pd.read_csv(PREVIOUS_REPLAY_PATH) if PREVIOUS_REPLAY_PATH.exists() else pd.DataFrame()
init_qlib(QLIB_DATA_PATH)

coverage_rows: list[dict[str, Any]] = []
fold_cache: dict[str, dict[str, Any]] = {}
for fold, spec in WALK_FORWARD_WINDOWS.items():
    target_fold = str(spec['target_fold'])
    fold_targets = targets[targets['fold'].astype(str).eq(target_fold)].copy()
    train_start, train_end = map(pd.Timestamp, spec['train'])
    test_start, test_end = map(pd.Timestamp, spec['test'])
    train_targets = fold_targets[(fold_targets['datetime'] >= train_start) & (fold_targets['datetime'] <= train_end)].copy()
    overlap = train_targets[(train_targets['datetime'] >= test_start) & (train_targets['datetime'] <= test_end)]
    dataset = build_dataset(
        train_seg=spec['train'],
        valid_seg=spec['valid'],
        test_seg=spec['test'],
        feature_packs=tuple(profile.feature.packs),
        profile_name=profile.name,
    )
    feature_full = datetime_index_frame(flatten_feature_frame(dataset.handler.feature_frame))
    price_full = datetime_index_series(dataset.handler.price_frame['price'])
    train_frame = feature_full.loc[spec['train'][0]:spec['train'][1]].copy()
    warm_seg = build_warm_start_replay_segment(feature_full.index, eval_seg=spec['test'], lookback=LOOKBACK)
    fold_cache[fold] = {
        'dataset': dataset,
        'feature_full': feature_full,
        'price_full': price_full,
        'train_frame': train_frame,
        'warm_seg': warm_seg,
        'targets': train_targets,
    }
    coverage_rows.append({
        'fold': fold,
        'target_fold': target_fold,
        'target_rows': int(len(train_targets)),
        'target_start': str(train_targets['datetime'].min().date()) if len(train_targets) else '',
        'target_end': str(train_targets['datetime'].max().date()) if len(train_targets) else '',
        'train_start': spec['train'][0],
        'train_end': spec['train'][1],
        'test_start': spec['test'][0],
        'test_end': spec['test'][1],
        'feature_start': str(feature_full.index.min().date()),
        'feature_end': str(feature_full.index.max().date()),
        'train_frame_rows': int(len(train_frame)),
        'warm_start': warm_seg[0],
        'warm_end': warm_seg[1],
        'eval_overlap_rows': int(len(overlap)),
        'fold_local_gate': bool(len(train_targets) > 0 and len(overlap) == 0),
    })
coverage = pd.DataFrame(coverage_rows)
coverage.to_csv(CARD / 'coverage.csv', index=False)
coverage


[98193:MainThread](2026-06-14 09:16:10,995) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[98193:MainThread](2026-06-14 09:16:11,404) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[98193:MainThread](2026-06-14 09:16:11,404) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/data/qlib/soxl')}


[98193:MainThread](2026-06-14 09:16:12,334) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | Loading data Done


[98193:MainThread](2026-06-14 09:16:12,335) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


...

[98193:MainThread](2026-06-14 09:16:13,396) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Init data Done


[98193:MainThread](2026-06-14 09:16:14,647) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | Loading data Done


[98193:MainThread](2026-06-14 09:16:14,648) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[98193:MainThread](2026-06-14 09:16:14,649) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | Init data Done


,fold,target_fold,target_rows,target_start,target_end,train_start,train_end,test_start,test_end,feature_start,feature_end,train_frame_rows,warm_start,warm_end,eval_overlap_rows,fold_local_gate
0,wf_2020,wf_2020_train,2470,2010-03-11,2019-12-31,2010-03-11,2019-12-31,2020-01-01,2020-12-31,2011-03-11,2020-12-31,2217,2019-10-07,2020-12-31,0,True
1,wf_2022,wf_2022_train,253,2020-01-02,2020-12-31,2010-03-11,2020-12-31,2022-01-01,2022-12-31,2011-03-11,2022-12-30,2470,2021-10-07,2022-12-30,0,True
2,wf_2023_2024,wf_2023_2024_train,756,2020-01-02,2022-12-30,2010-03-11,2022-12-31,2024-01-01,2024-12-31,2011-03-11,2024-12-31,2973,2023-10-05,2024-12-31,0,True


In [3]:

metric_rows: list[dict[str, Any]] = []
fit_rows: list[dict[str, Any]] = []
slice_frames: list[pd.DataFrame] = []
action_frames: list[pd.DataFrame] = []
drawdown_rows: list[dict[str, Any]] = []
daily_manifest: list[dict[str, Any]] = []

relative_targets = gate_contract.get('target_relative_return_by_fold_pp', {})
previous_by_fold = previous_replay.set_index('fold').to_dict('index') if not previous_replay.empty and 'fold' in previous_replay else {}

for fold, spec in WALK_FORWARD_WINDOWS.items():
    print(f"[fold:{fold}] train={spec['train']} test={spec['test']} target_fold={spec['target_fold']}")
    cache = fold_cache[fold]
    fold_targets = cache['targets']
    if fold_targets.empty:
        metric_rows.append({'fold': fold, 'status': 'skipped_no_targets'})
        continue
    set_seed(SEED + list(WALK_FORWARD_WINDOWS).index(fold) * 17)
    fit = fit_state_objective_bc_actor_from_frame(
        cache['train_frame'],
        targets=fold_targets,
        profile=profile,
        train_config=BC_TRAIN_CONFIG,
        sample_weight_config=BC_WEIGHT_CONFIG,
        sample_size=0,
        rng_seed=SEED,
    )
    fit_rows.append({
        'fold': fold,
        'target_fold': spec['target_fold'],
        'target_rows': int(len(fold_targets)),
        'bc_samples': int(len(fit.actions)),
        'unique_actions': int(pd.Series(fit.actions).nunique()) if len(fit.actions) else 0,
        **{k: float(v) if isinstance(v, (int, float, np.number)) and np.isfinite(float(v)) else v for k, v in fit.metrics.items()},
    })
    bc_dataset = build_state_objective_bc_dataset_from_frame(
        cache['train_frame'],
        targets=fold_targets,
        profile=profile,
        sample_size=0,
        rng_seed=SEED,
    )
    slice_metrics = evaluate_bc_actor_slice_fit(
        fit.actor,
        fit.obs,
        fit.actions,
        bc_dataset.metadata,
        slice_cols=SLICE_COLS,
    )
    slice_metrics.insert(0, 'fold', fold)
    slice_frames.append(slice_metrics)
    action_dist = fit.action_distribution.copy()
    action_dist.insert(0, 'fold', fold)
    action_frames.append(action_dist)

    bt = backtest_profile(
        fit.actor,
        cache['feature_full'],
        cache['price_full'],
        position_cap_config=profile.position_cap,
        position_budget_config=profile.position_budget,
        execution_config=profile.execution,
        mode_router_config=profile.mode_router,
        asset_idx=0,
        feature_columns=feature_columns,
        model_config=profile.model,
        test_seg=cache['warm_seg'],
    )
    daily = slice_warm_start_daily_path(
        portfolio_df=bt['portfolio'],
        price_series=cache['price_full'],
        eval_seg=spec['test'],
        position_history=bt.get('position_history', []),
        trade_signals=bt.get('trade_signals', []),
    )
    daily_path = CARD / f'riskcap_full_replacement_{fold}_official_daily_path.csv'
    daily.to_csv(daily_path, index=False)
    metrics = daily_path_metrics(daily)
    drawdown = max_drawdown_interval_metrics(daily)
    drawdown_rows.append({'fold': fold, **drawdown, 'daily_path': str(daily_path.relative_to(ROOT))})
    daily_manifest.append({
        'fold': fold,
        'path': str(daily_path.relative_to(ROOT)),
        'rows': int(len(daily)),
        'start': metrics.get('start'),
        'end': metrics.get('end'),
    })

    previous = previous_by_fold.get(fold, {})
    target_relative = float(relative_targets.get(fold, 30.0))
    return_floor = float(gate_contract.get('wf2020_reference_floor_return_pct', math.nan)) if fold == 'wf_2020' else float('-inf')
    max_dd_gate = bool(float(metrics['max_dd']) < TARGET_MAX_DD)
    relative_gate = bool(float(metrics['relative_return']) >= target_relative)
    return_floor_gate = bool(fold != 'wf_2020' or float(metrics['return']) >= return_floor)
    fold_gate = bool(max_dd_gate and relative_gate and return_floor_gate)
    metric_rows.append({
        'fold': fold,
        'status': 'evaluated',
        'target_fold': spec['target_fold'],
        'warm_start': cache['warm_seg'][0],
        'warm_end': cache['warm_seg'][1],
        'target_rows': int(len(fold_targets)),
        'bc_samples': int(len(fit.actions)),
        'return': float(metrics['return']),
        'buy_and_hold_return': float(metrics['buy_and_hold_return']),
        'relative_return': float(metrics['relative_return']),
        'max_dd': float(metrics['max_dd']),
        'avg_pos': float(metrics['avg_pos']),
        'trade_count': int(pd.Series(daily['action']).astype(str).str.upper().isin(['BUY', 'SELL']).sum()),
        'target_relative_return': target_relative,
        'target_max_dd': TARGET_MAX_DD,
        'wf2020_reference_floor_return': return_floor if math.isfinite(return_floor) else np.nan,
        'relative_gate': relative_gate,
        'max_dd_gate': max_dd_gate,
        'return_floor_gate': return_floor_gate,
        'fold_gate': fold_gate,
        'previous_selected_relative_return': previous.get('selected_relative_return', np.nan),
        'previous_selected_max_dd': previous.get('selected_max_dd', np.nan),
        'relative_delta_vs_previous': float(metrics['relative_return']) - float(previous.get('selected_relative_return', np.nan)) if previous else np.nan,
        'max_dd_delta_vs_previous': float(metrics['max_dd']) - float(previous.get('selected_max_dd', np.nan)) if previous else np.nan,
        'daily_path': str(daily_path.relative_to(ROOT)),
    })

fit_metrics = pd.DataFrame(fit_rows)
fold_metrics = pd.DataFrame(metric_rows)
slice_metrics = pd.concat(slice_frames, ignore_index=True) if slice_frames else pd.DataFrame()
action_distribution = pd.concat(action_frames, ignore_index=True) if action_frames else pd.DataFrame()
drawdown_intervals = pd.DataFrame(drawdown_rows)
daily_manifest_df = pd.DataFrame(daily_manifest)

fit_metrics.to_csv(CARD / 'bc_fit_metrics.csv', index=False)
fold_metrics.to_csv(CARD / 'fold_replay_metrics.csv', index=False)
slice_metrics.to_csv(CARD / 'bc_slice_metrics.csv', index=False)
action_distribution.to_csv(CARD / 'bc_action_distribution.csv', index=False)
drawdown_intervals.to_csv(CARD / 'drawdown_interval_diagnostic.csv', index=False)
daily_manifest_df.to_csv(CARD / 'daily_path_manifest.csv', index=False)

fold_metrics


[fold:wf_2020] train=('2010-03-11', '2019-12-31') test=('2020-01-01', '2020-12-31') target_fold=wf_2020_train


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


[98193:MainThread](2026-06-14 09:16:20,128) WARNING - qlib.data - [data.py:665] - load calendar error: freq=day, future=True; return current calendar!


[98193:MainThread](2026-06-14 09:16:20,128) WARNING - qlib.data - [data.py:668] - You can get future calendar by referring to the following document: https://github.com/microsoft/qlib/blob/main/scripts/data_collector/contrib/README.md


[98193:MainThread](2026-06-14 09:16:20,131) WARNING - qlib.BaseExecutor - [executor.py:121] - `common_infra` is not set for <qlib.backtest.executor.SimulatorExecutor object at 0x13adc8b90>


...

backtest loop:   0%|          | 0/311 [00:00<?, ?it/s]

/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rzi

/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rzi

/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rzi

,fold,status,target_fold,warm_start,warm_end,target_rows,bc_samples,return,buy_and_hold_return,relative_return,...,wf2020_reference_floor_return,relative_gate,max_dd_gate,return_floor_gate,fold_gate,previous_selected_relative_return,previous_selected_max_dd,relative_delta_vs_previous,max_dd_delta_vs_previous,daily_path
0,wf_2020,evaluated,wf_2020_train,2019-10-07,2020-12-31,2470,2156,14.057336,58.093571,-44.036235,...,69.604552,True,True,False,False,-277.735388,15.489488,233.699153,-0.616167,runtime/model_selection/card/wf2020_riskcap_fu...
1,wf_2022,evaluated,wf_2022_train,2021-10-07,2022-12-30,253,252,-10.707721,-86.588072,75.880351,...,NaN,True,True,True,True,65.864820,18.802284,10.015531,0.293991,runtime/model_selection/card/wf2020_riskcap_fu...
2,wf_2023_2024,evaluated,wf_2023_2024_train,2023-10-05,2024-12-31,756,755,15.256317,-0.356635,15.612952,...,NaN,False,True,True,False,35.566229,8.955769,-19.953277,7.100750,runtime/model_selection/card/wf2020_riskcap_fu...


In [4]:

evaluated = fold_metrics[fold_metrics['status'].eq('evaluated')].copy()
required_folds = list(WALK_FORWARD_WINDOWS)
missing_folds = [fold for fold in required_folds if fold not in set(evaluated['fold'])]
failed_folds = evaluated.loc[~evaluated['fold_gate'].astype(bool), 'fold'].tolist() if not evaluated.empty else required_folds
full_gate = bool((not evaluated.empty) and not missing_folds and evaluated['fold_gate'].astype(bool).all())
max_dd_pass_count = int(evaluated['max_dd_gate'].sum()) if not evaluated.empty else 0
relative_pass_count = int(evaluated['relative_gate'].sum()) if not evaluated.empty else 0
return_floor_pass_count = int(evaluated['return_floor_gate'].sum()) if not evaluated.empty else 0

wf2020_row = evaluated[evaluated['fold'].eq('wf_2020')].iloc[0].to_dict() if 'wf_2020' in set(evaluated['fold']) else {}
if full_gate:
    decision = 'wf2020_riskcap_full_bc_three_fold_gate_pass_no_promotion'
    next_mechanism = 'run_fixed_split_ytd_retention_training_and_actor_review_before_any_promotion'
elif wf2020_row and bool(wf2020_row.get('max_dd_gate')) and not bool(wf2020_row.get('return_floor_gate')):
    decision = 'wf2020_riskcap_full_bc_underparticipation_no_promotion'
    next_mechanism = 'design_return_preserving_hybrid_target_with_riskcap_floor_then_rerun_three_fold_bc_replay'
elif wf2020_row and not bool(wf2020_row.get('max_dd_gate')):
    decision = 'wf2020_riskcap_full_bc_drawdown_failed_no_promotion'
    next_mechanism = 'repair_riskcap_state_or_execution_threshold_before_retained_training'
else:
    decision = 'wf2020_riskcap_full_bc_partial_gate_no_promotion'
    next_mechanism = 'inspect_failed_fold_and_design_return_preserving_hybrid_target_before_ppo'

selected_by_fold = {
    str(row['fold']): {
        'fold': str(row['fold']),
        'status': str(row['status']),
        'selected_candidate': 'riskcap_full_replacement_bc_initialization',
        'selected_kind': 'bc_initialization',
        'selected_iter': 0,
        'selected_return': float(row['return']),
        'selected_relative_return': float(row['relative_return']),
        'selected_max_dd': float(row['max_dd']),
        'selected_avg_pos': float(row['avg_pos']),
        'selected_target_gate': bool(row['fold_gate']),
        'selected_candidate_gate': bool(row['fold_gate']),
        'selected_drift_gate': bool(row['max_dd_gate']),
        'target_relative_return': float(row['target_relative_return']),
        'target_max_dd': float(row['target_max_dd']),
        'daily_path': row['daily_path'],
    }
    for row in evaluated.to_dict('records')
}

selection = {
    'experiment': EXPERIMENT,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'decision': decision,
    'role': 'diagnose',
    'profile': PROFILE_NAME,
    'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
    'direct_comparison_baseline': 'wf2020_risk_cap_reference_target_screen_v1',
    'reference_baseline': 'soxlbacktest_qlib_buysell_reference:risk_cap_35_15_05',
    'closest_historical_success_case': 'bull_reentry_three_fold_checkpoint_metric_replay_v1 passed wf_2022 and wf_2023_2024 but failed wf_2020 return.',
    'closest_historical_failure_case': 'wf2020_reference_floor_gate_calibration_v1 showed the prior wf_2020 candidate stayed short of the calibrated risk-cap reference floor.',
    'why_not_duplicate': 'This card runs the selected riskcap_full_replacement target through file-backed official BC replay for all required walk-forward folds.',
    'changed_layer': ['model_selection', 'training_target', 'bc_warm_start', 'official_replay'],
    'target_contract': str(TARGETS_PATH.relative_to(ROOT)),
    'target_max_dd_pct': TARGET_MAX_DD,
    'target_relative_return_by_fold_pp': gate_contract.get('target_relative_return_by_fold_pp', {}),
    'required_wf_windows': required_folds,
    'folds_evaluated': evaluated['fold'].tolist() if not evaluated.empty else [],
    'missing_folds': missing_folds,
    'failed_folds': failed_folds,
    'fold_gate_pass_count': int(evaluated['fold_gate'].sum()) if not evaluated.empty else 0,
    'max_dd_pass_count': max_dd_pass_count,
    'relative_pass_count': relative_pass_count,
    'return_floor_pass_count': return_floor_pass_count,
    'selected_by_fold': selected_by_fold,
    'promotion_allowed': False,
    'actor_retention_allowed': False,
    'training_allowed_from_this_card': bool(full_gate),
    'next_training_allowed_after_replay': bool(full_gate),
    'promotion_blocker': 'Diagnostic BC official replay only. No retained actor bundle, fixed-split/YTD replay, or full promotion review is produced in this card.',
    'next_mechanism': next_mechanism,
    'source_decisions': {
        'wf2020_risk_cap_reference_target_screen_v1': source_summary.get('decision'),
        'bull_reentry_three_fold_checkpoint_metric_replay_v1': 'three_fold_checkpoint_metric_replay_partial_pass_no_promotion',
    },
    'sidecars': [
        'coverage.csv',
        'bc_fit_metrics.csv',
        'bc_slice_metrics.csv',
        'bc_action_distribution.csv',
        'fold_replay_metrics.csv',
        'drawdown_interval_diagnostic.csv',
        'daily_path_manifest.csv',
        'riskcap_full_replacement_<fold>_official_daily_path.csv',
        'selection.json',
        'summary.json',
        'conclusion.md',
    ],
}

summary = {
    'experiment': EXPERIMENT,
    'decision': decision,
    'created_at': selection['created_at'],
    'canonical_baseline': selection['canonical_baseline'],
    'direct_comparison_baseline': selection['direct_comparison_baseline'],
    'reference_baseline': selection['reference_baseline'],
    'role': selection['role'],
    'changed_layer': selection['changed_layer'],
    'promotion_allowed': False,
    'actor_retention_allowed': False,
    'training_allowed_from_this_card': bool(full_gate),
    'next_training_allowed_after_replay': bool(full_gate),
    'next_mechanism': next_mechanism,
    'target_contract': selection['target_contract'],
    'target_max_dd_pct': TARGET_MAX_DD,
    'target_relative_return_by_fold_pp': selection['target_relative_return_by_fold_pp'],
    'required_best_return_pct': gate_contract.get('required_ytd_return_pct'),
    'required_soxlbacktest_return_pct': gate_contract.get('required_soxlbacktest_reference_return_pct'),
    'wf2020_reference_floor_return_pct': gate_contract.get('wf2020_reference_floor_return_pct'),
    'wf2020_reference_floor_max_dd_pct': gate_contract.get('wf2020_reference_floor_max_dd_pct'),
    'required_wf_windows': required_folds,
    'missing_folds': missing_folds,
    'failed_folds': failed_folds,
    'selected_by_fold': selected_by_fold,
    'fold_gate_pass_count': selection['fold_gate_pass_count'],
    'max_dd_pass_count': max_dd_pass_count,
    'relative_pass_count': relative_pass_count,
    'return_floor_pass_count': return_floor_pass_count,
    'source_summary': {
        'decision': source_summary.get('decision'),
        'selected_variant': source_summary.get('selected_variant'),
        'selected_target_distribution': source_summary.get('selected_target_distribution'),
    },
    'source_decisions': selection['source_decisions'],
    'fold_results': evaluated.to_dict('records'),
    'bc_train_config': asdict(BC_TRAIN_CONFIG),
    'bc_sample_weight_config': asdict(BC_WEIGHT_CONFIG),
    'sidecars': selection['sidecars'],
    'conclusion': [],
}

if evaluated.empty:
    summary['conclusion'].append('No folds evaluated; do not continue this branch.')
else:
    for row in evaluated.to_dict('records'):
        summary['conclusion'].append(
            f"{row['fold']}: return={row['return']:.2f}%, relative={row['relative_return']:.2f}pp "
            f"(target {row['target_relative_return']:.2f}pp), MaxDD={row['max_dd']:.2f}% "
            f"(<{TARGET_MAX_DD:.2f}% gate={bool(row['max_dd_gate'])}), avg_pos={row['avg_pos']:.3f}, "
            f"fold_gate={bool(row['fold_gate'])}."
        )
    if full_gate:
        summary['conclusion'].append('All required folds passed the diagnostic BC replay gates, but no runtime promotion is allowed until fixed-split/YTD actor retention review is run.')
    else:
        summary['conclusion'].append('No runtime promotion: at least one required fold still fails the replay gate; use this evidence to design the next target rather than retaining an actor.')

write_json(CARD / 'selection.json', selection)
write_json(CARD / 'summary.json', summary)
conclusion_lines = [
    f'# {EXPERIMENT}',
    '',
    f'Decision: `{decision}`.',
    '',
    f'Fold gates passed: `{selection["fold_gate_pass_count"]}/{len(required_folds)}`. MaxDD gates passed: `{max_dd_pass_count}/{len(required_folds)}`. Relative gates passed: `{relative_pass_count}/{len(required_folds)}`.',
    '',
    *[f'- {line}' for line in summary['conclusion']],
    '',
    f'Next action: `{next_mechanism}`.',
    '',
    'Promotion note: `promotion_allowed=false`, `actor_retention_allowed=false`; this card is a diagnostic replay, not a retained model.',
]
(CARD / 'conclusion.md').write_text('\n'.join(conclusion_lines) + '\n', encoding='utf-8')
summary


{'experiment': 'wf2020_riskcap_full_bc_replay_v1',
 'decision': 'wf2020_riskcap_full_bc_underparticipation_no_promotion',
 'created_at': '2026-06-14T01:16:29.461682+00:00',
 'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
 'direct_comparison_baseline': 'wf2020_risk_cap_reference_target_screen_v1',
 'reference_baseline': 'soxlbacktest_qlib_buysell_reference:risk_cap_35_15_05',
 'role': 'diagnose',
 'changed_layer': ['model_selection',
  'training_target',
  'bc_warm_start',
  'official_replay'],
 'promotion_allowed': False,
 'actor_retention_allowed': False,
 'training_allowed_from_this_card': False,
 'next_training_allowed_after_replay': False,
 'next_mechanism': 'design_return_preserving_hybrid_target_with_riskcap_floor_then_rerun_three_fold_bc_replay',
 'target_contract': 'runtime/model_selection/card/wf2020_risk_cap_reference_target_screen_v1/riskcap_full_replacement_combined_target_contract.csv',
 'target_max_dd_pct': 20.0,
 'target_relative_return_by_fold_pp': {'wf_

## Conclusion

The executable result is recorded in `summary.json`, `selection.json`, and `conclusion.md`. Promotion remains blocked unless all required fold gates pass and a later retained actor review proves fixed-split, YTD/common-window return alignment, and dashboard replay evidence.